# Demo: Baseline Extraction

A prompt-only Research Agent attempts to emit Turtle directly, without `DatasetMiddleware`.

---

- **What condition does this notebook represent?** A baseline in which the agent receives task guidance but no dataset-backed tool surface, no middleware-owned RDF working memory, and no middleware-controlled serialization path.
- **What is the point of this baseline?** To establish what direct prompting can and cannot do before we add capability-gated middleware.
   - Can the agent produce parseable Turtle at all?
   - If it can, how much semantic overlap does it have with a target graph?
   - If it cannot, syntax failure itself is part of the result.
- **What is intentionally still present?** Deepagents' default behavior and `MinistralPromptSuffixMiddleware` remain so that we do not accidentally degrade the model for reasons unrelated to RDF extraction.
- **Where does this notebook fit in the series?** It is the negative control for [demo-dataset-middleware.ipynb](./demo-dataset-middleware.ipynb), where the Research Agent receives dataset tools and a middleware-owned RDF side channel.

## 1. Setup

We first filter a LangChain-internal warning that is unrelated to the experiment.

In [1]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.",
    category=UserWarning,
)

### 1.1. Bring your own Model

This notebook uses [LangChain's OpenAI integration](https://docs.langchain.com/oss/python/integrations/chat/openai) for [a self-hosted model provider](https://docs.langchain.com/langsmith/playground-model-providers#openai-compatible-endpoint), but you can use any of [LangChain's supported providers](https://docs.langchain.com/oss/python/integrations/providers/all_providers)
What's important is that Deepagents requires a chat model capable of tool invocations.

_NOTE:_ [My Ministral model](https://huggingface.co/mistralai/Ministral-3-14B-Reasoning-2512) recommends adding specific suffix to its prompt. In this repository we also provide `MinistralPromptSuffixMiddleware` to append it so that it doesn't catch people by surprise if they are using other models. Instead, just modify the prompt directly to match your specific model's requirements. Additionally, the model's documentation indicates that one should use `temperature=1` over `temperature=0`, so we follow that guidance as well.

In [2]:
import os

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv("CHAT_MODEL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    seed=42,
    temperature=1,
)

## 2. The Demonstration

This notebook uses the same task family as the middleware demos, but keeps the runtime condition deliberately narrow.

### 2.1. Prompt Setup

The shared prompt is assembled from reusable components in [`demo_utils.py`](./demo_utils.py).
Those components include RDF modeling guidance, vocabulary-selection guidance, and a shared stop condition.

Methodologically, these additions should be understood as **tool-agnostic guidance extracted from middleware prompts**.
The goal is to reduce prompt asymmetry between conditions, not to silently tune the baseline or recreate middleware capabilities.

What this baseline **does** receive:

- Domain-relevant instructions about RDF modeling.
- Guidance to prefer grounded, stable, documented modeling choices.
- A direct instruction to return exactly one Turtle block.

What this baseline **does not** receive:

- `DatasetMiddleware` tools such as `add_triples` or `serialize_dataset`.
- A middleware-owned RDF dataset that can accumulate state across reasoning steps.
- A structured serialization path that can prevent malformed final Turtle.

That distinction matters for interpretation: if `DatasetMiddleware` produces more parseable outputs, that is part of the middleware effect rather than evidence that the baseline was unfairly under-prompted.

### 2.2. Prompt Limitations

Prompt guidance can encourage better modeling choices, but it cannot replace missing affordances.
This baseline still asks the Research Agent to compose Turtle directly in its final answer.

As a result, the evaluation has a hard first gate:

1. Is there exactly one Turtle block?
2. Does that block parse as RDF?

If the answer to either question is no, semantic graph metrics are unavailable for that run.

In [3]:
from typing import Final

from demo_utils import CORE_PROMPT, DATASET_TIPS, STOPPING_CRITERIA, VOCABULARY_TIPS

# Reduce prompt asymmetry without recreating middleware-only capabilities.
SYSTEM_PROMPT: Final[str] = (
    CORE_PROMPT + DATASET_TIPS + VOCABULARY_TIPS + STOPPING_CRITERIA
)

### 2.3. Agent Task Definition

We keep the extraction task itself fixed across the series so that capability changes come from middleware composition rather than from changing the problem statement.

The task asks the Research Agent to represent a short natural-language passage as RDF.
In this baseline condition, the agent must do that by emitting Turtle directly in its final answer.

In [4]:
from demo_utils import TASK

print(TASK)


Please assist me in representing the subject matter of the following text as an RDF graph.
Please use <urn:ex:> as the base for any IRIs that you need to mint as part
of your response. Be sure to label any new terms or properties that you mint
so that they are human readable.

```text
John is a person. Modern people are classified as _homo sapiens_. Apparently,
homo sapiens falls under the subtribe of Hominina. Every Hominidae is a
Haplorhini, and now the names don't even sound like they're related. Then
we get to primates, which can be controversial for some people, for some reason.
Nobody argues with the idea that primates are mammals, yet some people take
umbrage with the idea that _homo sapiens_ is an animal. Oh, Hominidae contains
Hominina, too. Emotions and theological views aside, can we formalize this?
```



#### 2.3.1 Evaluation Contract

For this notebook, evaluation proceeds in two layers.

First, the final answer must be machine-usable:

1. The response should contain exactly one `text/turtle` fenced block.
2. That Turtle should parse successfully.

Second, if parsing succeeds, we compare the resulting graph against a target graph.
Best-practice documentation triples such as `rdfs:label` and `rdfs:comment` are acceptable extras rather than automatic errors.

In [5]:
from demo_utils import expected_graph
from rdflib import Graph

print(expected_graph.serialize(format="turtle"))

@prefix ex: <urn:ex:> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Haplorhini a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Primate .

ex:Hominidae a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Haplorhini .

ex:Hominina a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Hominidae .

ex:HomoSapiens a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Hominina .

ex:Mammal a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Animal .

ex:Primate a rdfs:Class,
        owl:Class ;
    rdfs:subClassOf ex:Mammal .

ex:John a ex:Person .

ex:Person rdfs:subClassOf ex:HomoSapiens ;
    owl:equivalentClass ex:HomoSapiens .




In [6]:
from deepagents import create_deep_agent
from langgraph.errors import GraphRecursionError
from rdflib_reasoning.middleware.ministral_middleware import (
    MinistralPromptSuffixMiddleware,
)
from rdflib_reasoning.middleware.tracing_notebook import LiveNotebookTrace

with LiveNotebookTrace(heading="Dataset Trace") as trace:
    agent = trace.attach(
        create_deep_agent(
            model=llm,
            system_prompt=SYSTEM_PROMPT,
            middleware=[
                MinistralPromptSuffixMiddleware(),
            ],
        )
    )

    RUN_INPUT = {"messages": [{"role": "user", "content": TASK}]}
    RUN_CONFIG = {"recursion_limit": 20}
    try:
        agent_response = agent.invoke(RUN_INPUT, config=RUN_CONFIG)
    except GraphRecursionError:
        display(trace.sink.snapshot())
        raise

# Dataset Trace

## Turn 1

### Final Response

> I will model the facts in the text about John, homo sapiens, Hominina, Hominidae, Haplorhini, and primates, explicitly asserting the listed scientific classifications and affiliations. I will not represent transient or emotional content, merely the factual, atemporal statements about taxa and relationships.
>
> Facts to model as RDF:
> - John is a person.
> - Modern people are classified as homo sapiens.
> - homo sapiens falls under the subtribe Hominina.
> - Every Hominidae is a Haplorhini.
> - Hominidae contains Hominina.
>
> I will define a new class `ex:Person` and think carefully about established terms:
> - `rdfs:Class` and `rdf:type` will be used for taxa and categories, but I will not mint new classes for species names unless necessary, as existing encodings often leave such labels as aliases or rdfs:label.
> - The text does not ask for uncertain or emotional modeling, so I will ignore "controversial", "comroversial", "emotions and theological views aside", etc.
> - I will mint `ex:isSubtribeOf` to represent the "falls under" relation, since SKOS Gary Paged List would be overkill and OWL subClassOf drinks down campanology.
> - I will assert `ex:isSubtribeOf` only where explicitly stated.
> - I will assert the Hominidae/Hominina containment as a subclass relation because the text says “contains” in the same sense as “falls under”.
>
> Step-by-step plan:
> 1. Define a namespace for our minted terms: `ex:` → `<urn:ex:>`.
> 2. Define a new property `ex:isSubtribeOf` with a clear label and comment.
> 3. Define a new class `ex:Person` with a label and comment.
> 4. Assert John as a `ex:Person`.
> 5. Assert that modern people are classified as a `homo_sapiens` (minted for consistency).
> 6. Assert `homo_sapiens` is a subtribe under `Hominina` via `ex:isSubtribeOf`.
> 7. Assert `Hominina` is a subtribe under `Hominidae` (since Hominidae contains Hominina).
> 8. Assert `Hominidae` is a subclass of `Haplorhini` via `owl:subClassOf`.
> 9. Prefer to reuse existing scientific names as blank nodes or instance URIs, but since the text treats them as distinct taxa, I will use IRIs with rdfs:label for human readability and rdf:type to宣一类。
>
> Let’s begin by defining the new terms and then the assertions.
>
> Add the following to the result graph:
>
> ```turtle
> @prefix ex: <urn:ex:> .
> @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
> @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
> @prefix owl: <http://www.w3.org/2002/07/owl#> .
> @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
>
> ### 1. Minted classes
> ex:Person a rdfs:Class ;
>     rdfs:label "Person"@en ;
>     rdfs:comment "A class for human individuals (modern people)."@en .
>
> ### 2. Minted property
> ex:isSubtribeOf a owl:ObjectProperty ;
>     rdfs:label "is subtribe of"@en ;
>     rdfs:comment "Relates a subtribe to its parent tribe (e.g. Hominina to Hominidae)."@en .
>
> ### 3. scientific taxa
> homo_sapiens a rdf:Class ;
>     rdfs:label "_homo sapiens_"@en ;
>     rdfs:comment "The scientific classification for modern people."@en .
>
> Hominidae a rdf:Class ;
>     rdfs:label "Hominidae"@en ;
>     rdfs:comment "The biological family (Hominidae), hominins and hominids."@en .
>
> Hominina a rdf:Class ;
>     rdfs:label "Hominina"@en ;
>     rdfs:comment "The subtribe (Hominina), under Hominidae."@en .
>
> Haplorhini a rdf:Class ;
>     rdfs:label "Haplorhini"@en ;
>     rdfs:comment "The order (Haplorhini), which includes Hominidae."@en .
>
> ### 4. assertions
> John a ex:Person ;
>     rdfs:label "John"@en .
>
> ex:Person rdfs:subClassOf homo_sapiens .
>
> homo_sapiens ex:isSubtribeOf Hominina .
>
> Hominina ex:isSubtribeOf Hominidae .
>
> Hominidae owl:subClassOf Haplorhini .
> ```
>
> Check:
> - All new terms are labeled.
> - All assertions are grounded in the explicit facts (e.g., “falls under” → `ex:isSubtribeOf`).
> - No speculativity or uncertain claims.
> - `owl:subClassOf` is appropriate for the assertion “Every Hominidae is a Haplorhini” (axiom).
> - `rdfs:subClassOf` is used where nicht expressed (modern people are homo sapiens).
> - “contains” is modeled as subclass membership.


### 2.4. Parseability Gate

Before we can talk about overlap, missing facts, or extra facts, we first need a graph.
The next cell extracts Turtle blocks from the final answer and asks RDFLib to parse them.

If parsing fails, that is not just a notebook inconvenience.
For an integrated system, a malformed final serialization is itself a failed output condition.


In [7]:
import mistune


def extract_turtle_code(markdown_text: str) -> list[str]:
    # 1. Initialize Mistune to return an Abstract Syntax Tree (AST)
    # Using 'ast' is much faster than rendering to HTML
    markdown = mistune.create_markdown(renderer="ast")
    tokens = markdown(markdown_text)

    turtle_blocks = []

    # 2. Iterate through tokens to find fenced code blocks
    for token in tokens:
        # Check if the block is a code block and has the 'turtle' info string
        if token["type"] == "block_code":
            if token.get("attrs", {}).get("info", None) in {"text/turtle", "turtle"}:
                turtle_blocks.append(token["raw"].strip())

    return turtle_blocks


def extract_graphs(markdown_text: str) -> list[Graph]:
    return [
        Graph().parse(data=turtle_block, format="turtle")
        for turtle_block in extract_turtle_code(markdown_text)
    ]


def extract_graph_union(markdown_text: str) -> Graph:
    return sum(extract_graphs(markdown_text), Graph())


message = agent_response["messages"][-1]
actual_graph: Graph | None = None
parse_error: Exception | None = None

try:
    actual_graph = extract_graph_union(message.content)
except Exception as e:
    parse_error = e
    print(f"Parseability gate failed: {e}")

Parseability gate failed: at line 18 of <>:
Bad syntax (expected directive or statement) at ^ in:
"...b'(e.g. Hominina to Hominidae)."@en .\n\n### 3. scientific taxa\n'^b'homo_sapiens a rdf:Class ;\n    rdfs:label "_homo sapiens_"@e'..."


### 2.5. Conditional Metrics

Semantic metrics are only meaningful after the parseability gate passes.
When we do have a valid graph, we can compare plain triple sets directly because this target graph contains no blank nodes.

That still leaves two interpretation issues:

1. Missing target triples usually indicate a modeling or grounding problem.
2. Extra triples are sometimes harmless documentation and sometimes genuine errors.

The following notes explain how to read those cases when a run produces parseable RDF.

In [8]:
from demo_utils import evaluate_actual_graph

intersection = Graph()
missing = Graph()
extra = Graph()
union = Graph()

if actual_graph is None:
    measures = None
    graphs = None
else:
    measures, graphs = evaluate_actual_graph(actual_graph)
    intersection = graphs["intersection"]
    missing = graphs["missing"]
    extra = graphs["extra"]
    union = graphs["union"]

In [9]:
import sys

if measures is None:
    print("Semantic metrics unavailable because the parseability gate failed.")
elif measures["counts"]["intersection"] > 0:
    print(intersection.serialize(format="turtle"))
else:
    print(
        "The agent's graph output did NOT intersect the expected graph.",
        file=sys.stderr,
    )

Semantic metrics unavailable because the parseability gate failed.


#### 2.5.1. Interpreting Missing Facts

When a parsed graph is missing target triples, several things may have happened.

- The Research Agent may have minted local terms instead of reusing the target vocabulary.
- It may have chosen a different but still partly intelligible modeling pattern.
- It may simply have omitted content that should have been represented.

These cases are not equivalent, which is why overlap metrics need qualitative interpretation alongside the counts.
They also motivate later middleware that helps the Research Agent inspect or retrieve vocabulary information instead of relying only on baseline model memory.

In [10]:
if measures is None:
    print("No missing-triples report because the parseability gate failed.")
else:
    print(missing.serialize(format="turtle"))

No missing-triples report because the parseability gate failed.


#### 2.5.2. Interpreting Extra Facts

Extra triples are not automatically bad.
A parsed graph may contain documentation triples or other defensible details that are absent from the minimal target graph.

At the same time, extra triples can also reveal genuine semantic mistakes, such as using a familiar RDF term incorrectly or reversing a relation's intended direction.
This is why simple set-based metrics are helpful but not sufficient on their own.

In [11]:
if measures is None:
    print("No extra-triples report because the parseability gate failed.")
else:
    print(extra.serialize(format="turtle"))

No extra-triples report because the parseability gate failed.


#### 2.5.3. What This Means for Evaluation

This baseline is useful, but only with explicit quality gates.

The practical evaluation order is:

1. Check whether the final Turtle block exists and parses.
2. If parsing succeeds, compute overlap metrics.
3. Interpret missing and extra triples manually enough to distinguish harmless variation from semantic failure.

The current checked-in run fails at step 1, which means semantic metrics are unavailable for that run.
That is still an informative result: direct serialization is brittle enough that parseability must be treated as its own baseline outcome.

This also clarifies the comparison with `DatasetMiddleware`.
When middleware moves graph construction into a structured side channel and serializes from middleware-owned state, improvements in syntax robustness and parseability are part of the measured benefit.


In [12]:
if measures is None:
    print("Semantic metrics unavailable because the parseability gate failed.")
else:
    expected_count = measures["counts"]["expected"]
    actual_count = measures["counts"]["extracted"]
    intersection_count = measures["counts"]["intersection"]
    missing_count = measures["counts"]["false_negatives"]
    extra_count = measures["counts"]["false_positives"]
    union_count = measures["counts"]["union_count"]

    precision = measures["metrics"]["precision"]
    recall = measures["metrics"]["recall"]
    f1_score = measures["metrics"]["f1_score"]
    jaccard_index = measures["metrics"]["jaccard_index"]
    triple_edit_distance = measures["metrics"]["triple_edit_distance"]
    normalized_triple_edit_distance = measures["metrics"][
        "normalized_triple_edit_distance"
    ]
    exact_match = measures["exact_match"]

    print(f"Expected triples: {expected_count}")
    print(f"Actual triples: {actual_count}")
    print(f"Intersection triples: {intersection_count}")
    print(f"Missing triples: {missing_count}")
    print(f"Extra triples: {extra_count}")
    print(f"Union triples: {union_count}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall: {recall:.3f}")
    print(f"F1 score: {f1_score:.3f}")
    print(f"Jaccard Index: {jaccard_index:.3f}")
    print(f"Triple Edit Distance: {triple_edit_distance}")
    print(f"Normalized Triple Edit Distance: {normalized_triple_edit_distance:.3f}")
    print(f"Exact Match: {exact_match}")

Semantic metrics unavailable because the parseability gate failed.


## 3. Conclusions

This notebook serves a narrow purpose: it shows what a prompt-only RDF extraction baseline looks like before dataset middleware is added.

In the checked-in run, the baseline does **not** clear the parseability gate.
That means we cannot compute semantic overlap metrics for this output, and that limitation is itself part of the result rather than an incidental notebook failure.

The prompt supplements used here are best understood as tool-agnostic guidance extracted from middleware prompts.
They reduce prompt asymmetry, but they do not reproduce the middleware's structured datastore side channel.

That is why improved syntax robustness in the middleware condition counts as a real positive result.
If `DatasetMiddleware` produces parseable RDF more reliably, that belongs in the comparison.

[The next notebook](./demo-dataset-middleware.ipynb) examines that tool-enabled condition directly.